# Suite MOD — Modules: schemas as data

A module is a named set of schemas held by an object of the meta-schema `Schemas.Module.Schema`, registered as
`Schemas.Module`. `Modules.module(schemas)` builds one and `Modules.schemas(module)` reads one back, so schemas are
written, read, validated and compared like any other objects. Within a module each schema is written inline, by kind,
and refers by name to the schemas in the module and those registered.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML
from support import raises, text

store = Proxies.OfStore()

B = store
Int32 = S.OfNative.resolve(lambda n: n.type(int).bits(32))
Size = S.OfNative.resolve(lambda n: n.token("ccpp", "size_t").bytes(8))
Phone = S.OfObject.Builder().properties(text("number")).create()
Reach = S.OfUnion.Builder().branches(lambda b: b.name("phone").of(Phone),
                                     lambda b: b.name("other").of(S.OfObject.Builder().properties(text("note")).create())
                                     ).create()
Stamped = S.OfIntersection.Builder().parts(lambda p: p.name("a").of(Phone), lambda p: p.name("b").of(Phone)).create()
Contact = S.OfObject.Builder().ref().properties(
    text("name"), lambda p: p.name("home").of(Phone), lambda p: p.name("reach").of(Reach),
    lambda p: p.name("stamped").of(Stamped), lambda p: p.name("ids").of(lambda t: t.as_indexed(lambda i: i.of(Int32)))
    ).create()
Directory = S.OfObject.Builder().ref().singleton("Directory").create()
Listing = S.OfRelation.Builder().links("directory", "contact").properties(text("key")).unique(
    "directory", "key").unique("contact").create()
Labels = S.OfRelation.Builder().links("contact", "phone").create()  # registered, not in the module, below
S.OfObject.Builder(Contact).relations(lambda a: a.name("listings").of(Listing).me("contact"),
                                      lambda a: a.name("labels").of(Labels).me("contact")).update()
S.OfObject.Builder(Directory).relations(lambda a: a.name("listings").of(Listing).me("directory")).update()
store.register("Labels", Labels)
SCHEMAS = {"Int32": Int32, "Size": Size, "Phone": Phone, "Contact": Contact, "Directory": Directory, "Listing": Listing}


def entries(module):
    """A module's entries, as plain data."""
    plain = Plain.ToPlain(store)(S.Module.Schema, module)
    return plain["objects"][plain["root"]].get("schemas", [])


def module_of(*entries):
    """A module decoded from plain entries, which `Modules.module` would not write."""
    return Plain.FromPlain(B)(S.Module.Schema, {"root": "s0", "objects": {"s0": {"schemas": list(entries)}}})


def named(name):
    return {"named": {"name": name}}


STR = {"native": {"format": "basic", "name": "str"}}

## MOD-01 · The meta-schemas validate, and `Schemas.Module` is registered as a reference object schema

In [ ]:
meta = [S.Module.Schema, S.Module.Entry, S.Module.Definition, S.OfAny.Schema, S.OfAny.Named, S.OfNative.Schema,
        S.OfProperty.Schema, S.OfObject.Schema, S.OfAdjacency.Schema, S.OfRelation.Schema, S.OfRelation.Ref,
        S.OfUnion.Schema, S.OfIntersection.Schema, S.OfIndexed.Schema]
assert all(schema.validate() == [] for schema in meta)
assert Modules.MODULE == "Schemas.Module" and store.schema("Schemas.Module") is S.Module.Schema and S.Module.Schema.ref
assert store.registered("Schemas.Module") is S.Module.Schema and store.registered("Labels") is Labels
with raises(LookupError, match="no schema registered as 'nope'"):
    store.registered("nope")
kinds = ["native", "object", "union", "intersection", "indexed"]
assert [b.name for b in S.OfAny.Schema.branches] == [*kinds, "named"]  # a type: a schema of any kind, or a name
assert [b.name for b in S.Module.Definition.branches] == [*kinds, "relation"]
assert [b.name for b in S.OfRelation.Ref.branches] == ["relation", "named"]
assert list(S.OfObject.Schema.properties) == ["properties", "adjacencies", "singleton", "ref"]
assert list(S.OfRelation.Schema.properties) == ["links", "properties", "uniques"]
assert list(S.OfNative.Schema.properties) == ["format", "name", "bits", "bytes"]

## MOD-02 · A module writes each schema inline, by kind, and refers by name to the schemas in it and those registered

In [ ]:
module = Modules.module(store, SCHEMAS)
assert module.schema_name() == "Schemas.Module" and Validators.Validate(B)(S.Module.Schema, module) == []
assert Modules.module(B, SCHEMAS) is not module
phone = {"object": {"properties": [{"name": "number", "type": STR}]}}
assert entries(module) == [
    {"name": "Int32", "schema": {"native": {"format": "basic", "name": "int", "bits": 32}}},
    {"name": "Size", "schema": {"native": {"format": "ccpp", "name": "size_t", "bytes": 8}}},
    {"name": "Phone", "schema": phone},
    {"name": "Contact", "schema": {"object": {
        "properties": [
            {"name": "name", "type": STR},
            {"name": "home", "type": named("Phone")},
            {"name": "reach", "type": {"union": {"branches": [
                {"name": "phone", "type": named("Phone")},
                {"name": "other", "type": {"object": {"properties": [{"name": "note", "type": STR}]}}}]}}},
            {"name": "stamped", "type": {"intersection": {"parts": [
                {"name": "a", "type": named("Phone")}, {"name": "b", "type": named("Phone")}]}}},
            {"name": "ids", "type": {"indexed": {"item": named("Int32")}}}],
        "adjacencies": [{"name": "listings", "relation": named("Listing"), "me": "contact"},
                        {"name": "labels", "relation": named("Labels"), "me": "contact"}],  # registered
        "ref": True}}},
    {"name": "Directory", "schema": {"object": {"adjacencies": [
        {"name": "listings", "relation": named("Listing"), "me": "directory"}], "singleton": "Directory", "ref": True}}},
    {"name": "Listing", "schema": {"relation": {"links": ["directory", "contact"], "properties": [{"name": "key", "type": STR}],
                                                "uniques": [["directory", "key"], ["contact"]]}}}]
# What has no name is written inline wherever it is used: a relation too. Empty contents are left out.
Pair = S.OfRelation.Builder().links("a", "b").create()
Lone = S.OfObject.Builder().relations(lambda a: a.name("pairs").of(Pair).me("a")).create()
assert entries(Modules.module(store, {"Lone": Lone, "Empty": S.OfUnion.Data()})) == [
    {"name": "Lone", "schema": {"object": {"adjacencies": [
        {"name": "pairs", "relation": {"relation": {"links": ["a", "b"]}}, "me": "a"}]}}},
    {"name": "Empty", "schema": {"union": {}}}]
assert entries(Modules.module(store, {})) == [] and Modules.schemas(store, getattr(B, "Schemas.Module")().create()) == {}
# The module is an object like any other: written and read as JSON and YAML, and compared.
text_ = JSON.ToJSON(store)(S.Module.Schema, module)
assert JSON.ToJSON(store)(S.Module.Schema, JSON.FromJSON(B)(S.Module.Schema, text_)) == text_
assert YAML.ToYAML(store)(S.Module.Schema, YAML.FromYAML(B)(S.Module.Schema, YAML.ToYAML(store)(S.Module.Schema, module))) == \
    YAML.ToYAML(store)(S.Module.Schema, module)
assert C.OfObject(S.Module.Schema, module).compare(C.OfObject(S.Module.Schema, Modules.module(store, SCHEMAS))) == 0
assert C.OfObject(S.Module.Schema, module).compare(C.OfObject(S.Module.Schema, Modules.module(store, {"Phone": Phone}))) is None

## MOD-03 · Reading a module back gives the same schemas, sharing what is named, recursive ones included

In [ ]:
module = Modules.module(store, SCHEMAS)
back = Modules.schemas(store, JSON.FromJSON(B)(S.Module.Schema, JSON.ToJSON(store)(S.Module.Schema, module)))
assert list(back) == list(SCHEMAS) and all(schema is not SCHEMAS[name] for name, schema in back.items())
assert all(schema.validate() == [] for schema in back.values())
assert JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back)) == JSON.ToJSON(store)(S.Module.Schema, module)
contact = back["Contact"]
assert contact.properties["home"] is back["Phone"] and contact.properties["reach"].branches[0].type is back["Phone"]
assert contact.properties["stamped"].parts[1].type is back["Phone"] and contact.properties["ids"].item is back["Int32"]
assert contact.adjacencies["listings"].relation is back["Listing"] is back["Directory"].adjacencies["listings"].relation
assert contact.adjacencies["labels"].relation is Labels  # resolved in the registry
assert contact.adjacencies["listings"].me == "contact" and contact.ref and back["Directory"].singleton == "Directory"
assert back["Int32"].type is int and back["Int32"].bits == 32 and back["Size"].type is None and back["Size"].bytes == 8
assert back["Listing"].links == ("directory", "contact") and back["Listing"].uniques == (
    frozenset({"directory", "key"}), frozenset({"contact"}))
# A schema that refers to itself is written once, by name, and read back as itself.
Node = S.OfObject.Builder().properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("children").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
tree = Modules.schemas(store, Modules.module(store, {"Node": Node}))["Node"]
assert tree.properties["children"].item is tree and tree.validate() == []
# Validating a schema that holds itself ends, and reports each problem once: as a property, an entry's or in a union.
S.OfObject.Builder(Node).properties(lambda p: p.name("w").of(lambda t: t.as_native(lambda n: n.type(int).bits(0)))).update()
assert Node.validate() == ["property 'w': a width in bits must be a positive int, got 0"]
assert S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("tree").of(tree)).create().validate() == []
Loop = S.OfUnion.Builder().create()
S.OfUnion.Builder(Loop).branches(lambda b: b.name("more").of(lambda t: t.as_indexed(lambda i: i.of(Loop))),
                                 lambda b: b.name("ints").of(lambda t: t.as_indexed(lambda i: i.of(Int32)))).update()
assert S.OfObject.Builder().properties(lambda p: p.name("loop").of(Loop)).create().validate() == []
# Keys and extents are written and read back.
Grid = S.OfObject.Builder().properties(
    lambda p: p.name("attrs").of(lambda t: t.as_indexed(lambda i: i.key(lambda t: t.as_native(str)).of(Int32))),
    lambda p: p.name("row").of(lambda t: t.as_indexed(lambda i: i.of(Int32).extent(1, 3))),
    lambda p: p.name("open").of(lambda t: t.as_indexed(lambda i: i.of(Int32).extent(2)))).create()
grid = Modules.schemas(store, Modules.module(store, {"Grid": Grid, "Int32": Int32}))["Grid"]
assert grid.properties["attrs"].key.type is str and grid.properties["attrs"].extent is None
assert grid.properties["row"].extent == S.OfIndexed.Extent(1, 3) and grid.properties["row"].key is None
assert grid.properties["open"].extent == S.OfIndexed.Extent(2, None)
written = Modules.schemas(store, module_of({"name": "L", "schema": {"indexed": {"item": STR, "extent": {"maximum": 3}}}}))["L"]
assert written.extent == S.OfIndexed.Extent(0, 3)  # a minimum left out is 0
# Inline schemas are read as new schemas of their kind.
Pair = S.OfRelation.Builder().links("a", "b").create()
Lone = S.OfObject.Builder().relations(lambda a: a.name("pairs").of(Pair).me("a")).create()
lone = Modules.schemas(store, Modules.module(store, {"Lone": Lone}))["Lone"]
assert lone.adjacencies["pairs"].relation.links == ("a", "b") and lone.adjacencies["pairs"].relation is not Pair

## MOD-04 · Refused: a schema referring to itself unnamed, an unknown name, a name of the wrong kind, a name defined twice

In [ ]:
Node = S.OfObject.Builder().properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("next").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
Holder = S.OfObject.Builder().properties(lambda p: p.name("root").of(Node)).create()
with raises(ValueError, match="a schema that refers to itself must be named, in the module or the store"):
    Modules.module(store, {"Holder": Holder})
with raises(TypeError, match="not a schema: 'text'"):
    Modules.module(store, {"Bad": "text"})
with raises(TypeError, match="unsupported native type 'weird'"):
    Modules.module(store, {"Bad": S.OfNative.Data("weird")})
property_named = lambda name: {"object": {"properties": [{"name": "p", "type": named(name)}]}}  # noqa: E731
with raises(LookupError, match="no schema named 'Nope' in the module or the store"):
    Modules.schemas(store, module_of({"name": "A", "schema": property_named("Nope")}))
with raises(TypeError, match="'Labels' is a relation, not a type"):
    Modules.schemas(store, module_of({"name": "A", "schema": property_named("Labels")}))
with raises(TypeError, match="'A' is not a relation"):
    Modules.schemas(store, module_of({"name": "A", "schema": {"object": {"adjacencies": [
        {"name": "x", "relation": named("A"), "me": "a"}]}}}))
with raises(ValueError, match="the module defines 'A' twice"):
    Modules.schemas(store, module_of({"name": "A", "schema": {"union": {}}}, {"name": "A", "schema": {"union": {}}}))